In [1]:
# Prediction = 0: transaction classified as legitimate.
#Prediction = 1: transaction classified as potentially fraudulent.
#1. Install dependencies if required
# Uncomment the following line in a fresh environment:
# !pip install tensorflow pandas numpy scikit-learn matplotlib seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)

from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# 2. Load the dataset

DATA_PATH = "creditcard.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        "creditcard.csv was not found. Place it in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)


In [ ]:
print("Dataset shape:", df.shape)

In [ ]:
display(df.head())

In [ ]:
# 3. Inspect the dataset

display(df.info())


In [ ]:
print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(10))


In [ ]:

print("\nClass distribution:")
display(df["Class"].value_counts())


In [ ]:
# 4. Visualize class imbalance

counts = df["Class"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(["Legitimate (0)", "Fraud (1)"], counts.values)
plt.title("Credit Card Transaction Class Distribution")
plt.ylabel("Number of Transactions")
plt.show()


In [ ]:

print(f"Fraud transactions: {df['Class'].sum():,}")


In [ ]:
print(f"Fraud percentage: {df['Class'].mean() * 100:.4f}%")


## 5. Prepare the data

The target column is `Class`. All remaining columns are predictors.

A stratified split is used because fraud is normally a very small minority class. The scaler is fitted only on the training data to avoid data leakage.


In [ ]:
# 5. Feature-target split

X = df.drop(columns=["Class"])
y = df["Class"].astype(int)



In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_train_full
)



In [ ]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


In [ ]:
# 6. Standardize features

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Number of input features:", X_train_scaled.shape[1])


## 7. Handle class imbalance

Accuracy can be misleading for fraud detection. We therefore use class weights so that errors on fraudulent transactions have greater influence during training.

We will evaluate using precision, recall, F1-score, ROC-AUC and especially PR-AUC.


In [ ]:
# Calculate balanced class weights

class_counts = np.bincount(y_train)
total = len(y_train)
n_classes = len(class_counts)

class_weights = {
    i: total / (n_classes * count)
    for i, count in enumerate(class_counts)
    if count > 0
}

print("Class weights:")
print(class_weights)


In [ ]:
# 8. Build the TensorFlow/Keras neural network

np.random.seed(42)


In [ ]:
tf.random.set_seed(42)



In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),

    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.30),

    layers.Dense(32, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.20),

    layers.Dense(16, activation="relu"),

    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="roc_auc"),
        keras.metrics.AUC(name="pr_auc", curve="PR")
    ]
)


In [ ]:

model.summary()


In [ ]:
# 9. Train the model

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_pr_auc",
    mode="max",
    patience=8,
    restore_best_weights=True
)



In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=2048,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
# 10. Plot training history

history_df = pd.DataFrame(history.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(history_df["pr_auc"], label="Training PR-AUC")
plt.plot(history_df["val_pr_auc"], label="Validation PR-AUC")
plt.xlabel("Epoch")
plt.ylabel("PR-AUC")
plt.title("Training vs Validation PR-AUC")
plt.legend()
plt.show()


In [ ]:
# 11. Evaluate the model

results = model.evaluate(
    X_test_scaled,
    y_test,
    batch_size=4096,
    verbose=0,
    return_dict=True
)

for metric, value in results.items():
    print(f"{metric}: {value:.4f}")


In [ ]:
# 12. Test-set predictions

y_prob = model.predict(
    X_test_scaled,
    batch_size=4096,
    verbose=0
).ravel()


In [ ]:

DEFAULT_THRESHOLD = 0.50
y_pred = (y_prob >= DEFAULT_THRESHOLD).astype(int)



In [ ]:
print(classification_report(
    y_test,
    y_pred,
    target_names=["Legitimate", "Fraud"],
    digits=4,
    zero_division=0
))



In [ ]:
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print("PR-AUC :", round(average_precision_score(y_test, y_prob), 4))


In [ ]:
# 13. Confusion matrix

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Legitimate", "Fraud"],
    yticklabels=["Legitimate", "Fraud"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Threshold 0.50")
plt.show()


In [ ]:
# 14. ROC curve

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()


In [ ]:
# 15. Precision-Recall curve

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()


## 16. Tune the fraud threshold

For fraud detection, 0.50 is not always the best threshold.

Here we select the threshold that maximizes validation-set F1-score. In a production banking system, the threshold should instead be selected using the business cost of false positives, false negatives, manual reviews and customer friction.


In [ ]:
# Find an F1-oriented threshold using validation data

val_prob = model.predict(
    X_val_scaled,
    batch_size=4096,
    verbose=0
).ravel()

val_precision, val_recall, val_thresholds = precision_recall_curve(
    y_val, val_prob
)

f1_scores = (
    2 * val_precision[:-1] * val_recall[:-1]
    / (val_precision[:-1] + val_recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_scores)
best_threshold = float(val_thresholds[best_idx])

print("Best validation threshold:", round(best_threshold, 4))
print("Validation precision:", round(float(val_precision[best_idx]), 4))
print("Validation recall:", round(float(val_recall[best_idx]), 4))
print("Validation F1:", round(float(f1_scores[best_idx]), 4))


In [ ]:
# 17. Evaluate test data with tuned threshold

y_pred_tuned = (y_prob >= best_threshold).astype(int)

print(classification_report(
    y_test,
    y_pred_tuned,
    target_names=["Legitimate", "Fraud"],
    digits=4,
    zero_division=0
))

cm_tuned = confusion_matrix(y_test, y_pred_tuned)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_tuned,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Legitimate", "Fraud"],
    yticklabels=["Legitimate", "Fraud"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Tuned Threshold = {best_threshold:.4f}")
plt.show()


In [ ]:
# 18. Save the trained Keras model

MODEL_PATH = "credit_card_fraud_keras_model.keras"
model.save(MODEL_PATH)

print(f"Model saved to: {MODEL_PATH}")


In [ ]:
# 19. Save preprocessing parameters

# This CSV stores the scaler parameters so the same transformation
# can be reproduced later.
scaler_params = pd.DataFrame({
    "feature": X.columns,
    "mean": scaler.mean_,
    "scale": scaler.scale_
})

SCALER_PATH = "credit_card_scaler_params.csv"
scaler_params.to_csv(SCALER_PATH, index=False)

print(f"Scaler parameters saved to: {SCALER_PATH}")


In [ ]:
# 20. Example inference function

def predict_fraud(transaction_df, threshold=best_threshold):
    """Predict fraud probability for one or more transactions.

    transaction_df must contain the same predictor columns as X,
    excluding the Class target column.
    """
    transaction_df = transaction_df[X.columns]

    scaled = scaler.transform(transaction_df)

    probabilities = model.predict(
        scaled,
        verbose=0
    ).ravel()

    predictions = (probabilities >= threshold).astype(int)

    return pd.DataFrame({
        "fraud_probability": probabilities,
        "prediction": predictions
    })

# Example:
# sample = X_test.iloc[[0]]
# print(predict_fraud(sample))


## 21. Interpretation

**Prediction = 0:** transaction classified as legitimate.

**Prediction = 1:** transaction classified as potentially fraudulent.

Important metrics:
- **Precision:** Of transactions flagged as fraud, how many were actually fraud?
- **Recall:** Of all actual fraud transactions, how many did the model detect?
- **F1-score:** Balance between precision and recall.
- **ROC-AUC:** Ranking/discrimination ability across thresholds.
- **PR-AUC:** Particularly useful for highly imbalanced fraud datasets.

### Production considerations
This notebook is suitable as an educational/project baseline. A production fraud system would additionally require temporal validation, leakage checks, drift monitoring, threshold/cost optimization, explainability, security controls, and ongoing model retraining.
